# Stage 3: The "That's What She Said" (TWSS) Detector
In this notebook, we load the `michael_scott_sft.pt` backbone and attach two lightweight linear heads to the final hidden state. By training on 187 diverse minimal pairs and mining hard-negative "bait" lines from the show transcripts, we force the model to learn the structural semantics of a double entendre rather than just memorizing vocabulary.

In [1]:
!pip install -q tokenizers schrutepy pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.


In [2]:
import os
import json
import random
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from schrutepy import schrutepy

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = Tokenizer.from_file("michael_scott_tokenizer.json")
EOT_TOKEN = tokenizer.token_to_id("<|endoftext|>")

def encode_batch(texts, max_len=64):
    ids_list, lengths = [], []
    for t in texts:
        ids = tokenizer.encode(t).ids[:max_len]
        lengths.append(len(ids))
        ids_list.append(ids + [EOT_TOKEN] * (max_len - len(ids)))
    return torch.tensor(ids_list, dtype=torch.long, device=device), torch.tensor(lengths, dtype=torch.long, device=device)

Using device: cuda


In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.c_attn = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.c_proj = nn.Linear(n_embed, n_embed, bias=False)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        attn_weights = None
        if return_attn:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
            causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            att = att.masked_fill(~causal_mask, float("-inf"))
            attn_weights = F.softmax(att, dim=-1)
            y = attn_weights @ v
        else:
            y = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True,
            )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y), attn_weights

class FeedForward(nn.Module):
    def __init__(self, n_embed: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embed: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed, n_head, dropout=dropout)
        self.ln2 = nn.LayerNorm(n_embed)
        self.ffwd = FeedForward(n_embed, dropout=dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        attn_out, attn_weights = self.attn(self.ln1(x), return_attn=return_attn)
        x = x + attn_out
        x = x + self.ffwd(self.ln2(x))
        return x, attn_weights

class MichaelScottTWSS(nn.Module):
    def __init__(self, vocab_size: int = 2048, n_embed: int = 256, n_head: int = 8, n_layer: int = 6, block_size: int = 256, dropout: float = 0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embed, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight

        self.twss_head = nn.Sequential(
            nn.LayerNorm(n_embed),
            nn.Linear(n_embed, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.proj_head = nn.Sequential(
            nn.Linear(n_embed, 128),
            nn.GELU(),
            nn.Linear(128, 64),
        )

    def forward(self, idx: torch.Tensor, lengths: torch.Tensor, return_attn: bool = False):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.drop(self.token_embedding(idx) + self.position_embedding(pos))

        all_attns = []
        for block in self.blocks:
            x, attn_weights = block(x, return_attn=return_attn)
            if return_attn:
                all_attns.append(attn_weights)

        x = self.ln_f(x)
        last_idx = (lengths - 1).clamp(min=0, max=T - 1)
        h_last = x[torch.arange(B, device=idx.device), last_idx]

        logit = self.twss_head(h_last).squeeze(-1)
        proj = F.normalize(self.proj_head(h_last), dim=-1)
        return logit, proj, all_attns

In [5]:
with open("diverse_minimal_pairs.json", "r") as f:
    ALL_MINIMAL_PAIRS = json.load(f)

random.seed(42)
shuffled_pairs = ALL_MINIMAL_PAIRS.copy()
random.shuffle(shuffled_pairs)

train_pairs = shuffled_pairs[:160]
val_pairs = shuffled_pairs[160:]

df = schrutepy.load_schrute().dropna(subset=["character", "text"]).reset_index(drop=True)

BAIT_REGEX = re.compile(
    r"\b(big|hard|harder|long|longer|fit|fits|wet|deep|fast|slow|stiff|tight|huge|"
    r"hole|stick|blow|push|pull|finish|finished|satisfied|mouth|hands|knees|bed|"
    r"sore|come|coming)\b", re.IGNORECASE
)
PRONOUN_REGEX = re.compile(r"\b(it|that|this)\b", re.IGNORECASE)
TWSS_REGEX = re.compile(r"what she said", re.IGNORECASE)

real_show_twss, bait_negatives, pronoun_negatives, general_negatives = [], [], [], []

for i in range(1, len(df)):
    prev_row, curr_row = df.iloc[i - 1], df.iloc[i]
    if (prev_row["season"], prev_row["episode"]) != (curr_row["season"], curr_row["episode"]):
        continue

    prev_char, curr_char = str(prev_row["character"]).strip(), str(curr_row["character"]).strip()
    prev_text, curr_text = str(prev_row["text"]).strip(), str(curr_row["text"]).strip()

    if TWSS_REGEX.search(curr_text) and prev_char != "Michael":
        if 3 <= len(prev_text.split()) <= 22:
            real_show_twss.append(prev_text)
        continue

    if curr_char == "Michael" and prev_char != "Michael" and not TWSS_REGEX.search(prev_text):
        words = prev_text.split()
        if not (4 <= len(words) <= 20):
            continue
        if BAIT_REGEX.search(prev_text):
            bait_negatives.append(prev_text)
        elif len(words) <= 13 and PRONOUN_REGEX.search(prev_text):
            pronoun_negatives.append(prev_text)
        else:
            general_negatives.append(prev_text)

real_show_twss = list(dict.fromkeys(real_show_twss))
bait_negatives = list(dict.fromkeys(bait_negatives))
pronoun_negatives = list(dict.fromkeys(pronoun_negatives))
general_negatives = list(dict.fromkeys(general_negatives))

random.seed(42)
random.shuffle(bait_negatives)
random.shuffle(pronoun_negatives)
random.shuffle(general_negatives)

val_hard_negs = bait_negatives[:40] + pronoun_negatives[:20]
train_hard_negs = bait_negatives[40:280] + pronoun_negatives[20:180] + general_negatives[:200]

print(f"Diverse Minimal Pairs Loaded: {len(ALL_MINIMAL_PAIRS)}")
print(f"Training Hard Negatives: {len(train_hard_negs)}")
print(f"Zero-Shot Canonical Show TWSS Setups: {len(real_show_twss)}")

Diverse Minimal Pairs Loaded: 187
Training Hard Negatives: 600
Zero-Shot Canonical Show TWSS Setups: 18


In [6]:
def supcon_loss(features: torch.Tensor, labels: torch.Tensor, temperature: float = 0.15) -> torch.Tensor:
    sim = torch.matmul(features, features.T) / temperature
    sim_max, _ = torch.max(sim, dim=1, keepdim=True)
    logits = sim - sim_max.detach()

    B = features.size(0)
    self_mask = torch.eye(B, dtype=torch.bool, device=device)
    pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)) & (~self_mask)

    exp_logits = torch.exp(logits) * (~self_mask).float()
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)

    pos_per_row = pos_mask.sum(dim=1).clamp(min=1)
    loss = -(pos_mask.float() * log_prob).sum(dim=1) / pos_per_row
    return loss.mean()

In [8]:
twss_model = MichaelScottTWSS(vocab_size=tokenizer.get_vocab_size()).to(device)

raw_ckpt = torch.load("michael_scott_sft.pt", map_location=device, weights_only=True)
sft_state = raw_ckpt["model_state_dict"] if "model_state_dict" in raw_ckpt else raw_ckpt
twss_model.load_state_dict(sft_state, strict=False)

for name, param in twss_model.named_parameters():
    if any(k in name for k in ["token_embedding", "position_embedding", "blocks.0.", "blocks.1.", "blocks.2.", "blocks.3."]):
        param.requires_grad = False

optimizer = torch.optim.AdamW(
    [p for p in twss_model.parameters() if p.requires_grad],
    lr=2.5e-4,
    weight_decay=0.05,
)

print("Starting TWSS training...")
for epoch in range(1, 101):
    twss_model.train()

    pair_batch = random.sample(train_pairs, 40)
    hard_batch = random.sample(train_hard_negs, 80)

    pos_texts = [p[0] for p in pair_batch]
    neg_texts = [p[1] for p in pair_batch]
    all_texts = pos_texts + neg_texts + hard_batch

    x_all, len_all = encode_batch(all_texts)
    n_p = len(pos_texts)
    labels = torch.tensor([1.0] * n_p + [0.0] * (n_p + len(hard_batch)), device=device)

    logits, projs, _ = twss_model(x_all, len_all)
    z_pos = logits[:n_p]
    z_pair_neg = logits[n_p : 2 * n_p]
    z_hard_neg = logits[2 * n_p :]

    asym_pair_loss = (
        F.binary_cross_entropy_with_logits(z_pos, torch.ones_like(z_pos))
        + 2.5 * F.binary_cross_entropy_with_logits(z_pair_neg, torch.zeros_like(z_pair_neg))
    )
    hard_neg_loss = 2.0 * F.binary_cross_entropy_with_logits(z_hard_neg, torch.zeros_like(z_hard_neg))
    pair_margin = F.softplus(z_pair_neg - z_pos + 1.0).mean()
    contrastive = supcon_loss(projs, labels)

    loss = asym_pair_loss + hard_neg_loss + 0.5 * pair_margin + 0.3 * contrastive

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(twss_model.parameters(), 1.0)
    optimizer.step()

    if epoch % 25 == 0:
        twss_model.eval()
        with torch.no_grad():
            vx, vlen = encode_batch([p[0] for p in val_pairs] + [p[1] for p in val_pairs])
            vlogits, _, _ = twss_model(vx, vlen)
            vp = torch.sigmoid(vlogits[: len(val_pairs)])
            vn = torch.sigmoid(vlogits[len(val_pairs) :])
            pair_acc = (vp > vn).float().mean().item()

            hx, hlen = encode_batch(val_hard_negs)
            hprobs = torch.sigmoid(twss_model(hx, hlen)[0])
            fpr = (hprobs > 0.65).float().mean().item()

            sx, slen = encode_batch(real_show_twss)
            sprobs = torch.sigmoid(twss_model(sx, slen)[0])
            show_recall = (sprobs > 0.65).float().mean().item()

        print(
            f"Epoch {epoch:3d}/100 | Loss: {loss.item():.4f} | "
            f"Pair Acc: {pair_acc*100:5.1f}% | "
            f"Hard-Neg FPR: {fpr*100:4.1f}% | "
            f"Show Recall: {show_recall*100:4.1f}%"
        )

Starting TWSS training...
Epoch  25/100 | Loss: 2.0910 | Pair Acc: 100.0% | Hard-Neg FPR:  3.3% | Show Recall:  5.6%
Epoch  50/100 | Loss: 1.5659 | Pair Acc: 100.0% | Hard-Neg FPR:  3.3% | Show Recall:  0.0%
Epoch  75/100 | Loss: 1.5761 | Pair Acc: 100.0% | Hard-Neg FPR:  3.3% | Show Recall:  0.0%
Epoch 100/100 | Loss: 1.5218 | Pair Acc: 100.0% | Hard-Neg FPR:  3.3% | Show Recall:  0.0%


In [9]:
twss_model.eval()
print("\n=== ZERO-SHOT REAL SHOW TWSS SETUPS ===")
with torch.no_grad():
    sx, slen = encode_batch(real_show_twss)
    sprobs = torch.sigmoid(twss_model(sx, slen)[0]).cpu().tolist()
    for line, p in list(zip(real_show_twss, sprobs))[:10]:
        tag = "TWSS!" if p > 0.65 else "miss "
        print(f"[{tag} | P={p:.2f}] {line}")

os.makedirs("weights", exist_ok=True)
torch.save(twss_model.state_dict(), "weights/michael_scott_twss.pt")
print("\nSaved robust v2 detector weights to weights/michael_scott_twss.pt!")


=== ZERO-SHOT REAL SHOW TWSS SETUPS ===
[miss  | P=0.00] No, thanks. I'm good.
[miss  | P=0.00] You already did me.
[miss  | P=0.01] But we invited guests.
[miss  | P=0.00] Are they going to call the cops?
[miss  | P=0.00] Yeah, I'm definitely gonna go alone.
[miss  | P=0.00] Come on guys, get out of here!
[miss  | P=0.01] Well, I don't think I'll be here in ten years.
[miss  | P=0.00] And you were directly under her the entire time?
[miss  | P=0.03] And... go. Force it in as deep as you can.
[miss  | P=0.00] It's the only gavel I could find.

Saved robust v2 detector weights to weights/michael_scott_twss.pt!
